# ECON 3916: ML Prediction Project — Final Project

Ian Solberg

April 15, 2026

Professor Piao

**From Question to Recommendation**

This notebook scaffolds your final project. Work through each part sequentially. By Week 12, this notebook (plus your `app.py` and report) will form your complete submission.

**AI Policy:** AI co-pilot is REQUIRED. Document every AI interaction in Part 7 (AI Methodology Appendix) using the P.R.I.M.E. framework.

---

## Part 0: Setup

In [1]:
# ============================================================
# Part 0: Setup — Run this cell first
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('Setup complete.')

Setup complete.


---
## Part 1: Problem Statement

Fill in each blank below. This becomes the opening paragraph of your report.

**My prediction question is:** ___

**This is a prediction (umbrella) problem because:** ___
(Reminder: prediction asks "can we forecast Y from X?" — not "does X cause Y?")

**The decision this enables:** ___
(Who is the stakeholder? What action would they take differently with your prediction?)

**Dataset:** FRED api data dump - Jan 2000 -> present, weekly with lags and scoring columns applied 
- **Source:** FREDapi via FRED_Loader custom module developed for easy data dumps in research by myself.
- **N =** 1918 weeks
- **Features =** 216 series
- **Target variable =** Median Household Income
- **Access date:** April 15, 2026 (Most recent Friday is final row)

---
## Part 2: Data Loading + EDA

### 2.1 Load Your Data

In [3]:
# ============================================================
# 2.1 Load your dataset
# ============================================================
# Replace the URL/path below with your data source

df = pd.read_csv("data/fred_data.csv")  # Example: local file
df.drop(columns=['Unnamed: 0'], inplace=True)  # Drop unnecessary index column if it exists
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Shape: (1918, 288)
Columns: ['date', 'Headline_CPI', 'Core_CPI', 'CPI_Food', 'CPI_Energy', 'CPI_Shelter', 'CPI_Medical_Services', 'CPI_Gasoline', 'CPI_Transportation', 'CPI_Apparel', 'CPI_Education', 'Sticky_CPI', 'Core_Sticky_CPI', 'Median_CPI', 'Trimmed_Mean_CPI_16pct', 'PCE_Price_Index', 'Core_PCE', 'Trimmed_Mean_PCE_12mo', 'Core_PCE_Quarterly', 'PPI_All_Commodities', 'PPI_Final_Demand', 'PPI_Final_Demand_Less_FE', 'PPI_Finished_Goods', 'PPI_Intermediate_Materials', 'Import_Price_Index_All', 'Export_Price_Index_All', 'UMich_Inflation_Expectations', 'Breakeven_Inflation_5Y', 'Breakeven_Inflation_10Y', 'Forward_Inflation_5Y5Y', 'Expected_Inflation_1Y_CleveFed', 'Expected_Inflation_10Y_CleveFed', 'Nominal_GDP', 'Real_GDP', 'Real_GDP_Per_Capita', 'Potential_GDP', 'Gross_Domestic_Income', 'Corporate_Profits_After_Tax', 'National_Income', 'GDP_Deflator', 'Industrial_Production', 'Capacity_Utilization', 'Durable_Goods_Orders', 'Manufacturers_New_Orders', 'Manufacturers_Total_Orders', 'Tota

,date,Headline_CPI,Core_CPI,CPI_Food,CPI_Energy,CPI_Shelter,CPI_Medical_Services,CPI_Gasoline,CPI_Transportation,CPI_Apparel,...,Curve_10Y2Y_Lag_52w,Curve_10Y3M_Lag_52w,Breakeven_5Y_Lag_26w,Breakeven_5Y_Lag_52w,Home_Price_YoY_Lag_78w,Credit_Impulse_Lag_13w,Credit_Impulse_Lag_26w,FFR_Chg_52w,FFR_Chg_52w_Lag_52w,Sentiment_Lag_13w
0,2000-01-07,169.3,179.3,165.6,115.0,190.6,260.1,115.9,145.1,130.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-14,169.3,179.3,165.6,115.0,190.6,260.1,115.9,145.1,130.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-21,169.3,179.3,165.6,115.0,190.6,260.1,115.9,145.1,130.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-28,169.3,179.3,165.6,115.0,190.6,260.1,115.9,145.1,130.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-02-04,170.0,179.4,166.2,118.8,190.9,261.3,119.8,145.9,130.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.2 Basic Description

In [10]:
# ============================================================
# 2.2 Describe your data
# ============================================================

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1918 entries, 0 to 1917
Columns: 288 entries, date to Sentiment_Lag_13w
dtypes: float64(272), int64(15), str(1)
memory usage: 4.2 MB


In [11]:
df.describe()

,Headline_CPI,Core_CPI,CPI_Food,CPI_Energy,CPI_Shelter,CPI_Medical_Services,CPI_Gasoline,CPI_Transportation,CPI_Apparel,CPI_Education,...,Curve_10Y2Y_Lag_52w,Curve_10Y3M_Lag_52w,Breakeven_5Y_Lag_26w,Breakeven_5Y_Lag_52w,Home_Price_YoY_Lag_78w,Credit_Impulse_Lag_13w,Credit_Impulse_Lag_26w,FFR_Chg_52w,FFR_Chg_52w_Lag_52w,Sentiment_Lag_13w
count,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,1918.000000,...,1866.000000,1866.000000,1736.000000,1710.000000,1788.000000,1879.000000,1866.000000,1866.000000,1814.000000,1905.000000
mean,261.831430,266.655202,269.215497,240.681937,320.576086,505.719338,261.636245,224.237483,127.578754,243.787911,...,0.912008,1.157340,2.150487,2.143652,3.595183,-0.069367,-0.069850,-0.072529,-0.074609,74.604934
std,56.120498,55.319668,64.134761,62.856763,83.825685,130.413226,73.007415,49.591884,6.376999,67.852814,...,0.863157,1.202954,0.558397,0.559849,6.176718,4.411072,4.426415,1.292805,1.311155,16.384551
min,169.300000,179.300000,165.600000,113.500000,190.600000,260.100000,99.200000,144.300000,114.244000,110.200000,...,-0.996000,-1.860000,-2.120000,-2.120000,-12.579241,-17.913638,-17.913638,-4.580000,-4.580000,50.000000
25%,215.445000,218.253000,217.813000,197.246000,248.128000,393.285000,208.300000,186.282000,121.100000,187.298000,...,0.502000,0.570000,1.784000,1.774000,0.000000,-0.992579,-1.004604,-0.250000,-0.260000,56.600000
50%,250.792000,257.145000,252.867000,242.714500,306.808000,516.807000,280.015000,209.979000,126.693000,256.679000,...,0.510000,0.583000,2.316000,2.302000,3.418339,0.000000,0.000000,0.000000,0.000000,73.500000
75%,330.293000,334.165000,346.603000,314.019000,424.069000,648.945000,335.500000,284.573000,135.804000,315.336000,...,1.557000,2.065500,2.600000,2.600000,7.066867,0.579094,0.582700,0.220000,0.250000,90.000000
max,330.293000,334.165000,346.622000,330.456000,424.069000,648.945000,409.671000,284.573000,135.804000,315.336000,...,2.868000,3.782000,3.554000,3.554000,20.719370,19.230344,19.230344,4.500000,4.500000,112.000000


### 2.3 Missing Data Assessment

In [ ]:
# ============================================================
# 2.3 Missing data heatmap (Ch 1: MCAR/MAR/MNAR)
# ============================================================

# missing_pct = df.isnull().mean().sort_values(ascending=False)
# print('Missing data (%) by column:')
# print(missing_pct[missing_pct > 0])

# # Visual: missing data heatmap
# plt.figure(figsize=(12, 6))
# sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
# plt.title('Missing Data Heatmap')
# plt.tight_layout()
# plt.show()

**Missing data strategy:** ___ 
(Is this MCAR, MAR, or MNAR? What will you do — drop, impute, or flag?)

### 2.4 Distribution Plots

In [ ]:
# ============================================================
# 2.4 Distribution of key features (Ch 3)
# ============================================================

# # Plot distributions for your most important features
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# 
# sns.histplot(df['feature_1'], kde=True, ax=axes[0])
# axes[0].set_title('Distribution of Feature 1')
# 
# sns.histplot(df['feature_2'], kde=True, ax=axes[1])
# axes[1].set_title('Distribution of Feature 2')
# 
# sns.histplot(df['target'], kde=True, ax=axes[2])
# axes[2].set_title('Distribution of Target Variable')
# 
# plt.tight_layout()
# plt.show()

### 2.5 Outlier Detection

In [ ]:
# ============================================================
# 2.5 Outlier detection (Ch 4: Tukey Fences / IQR)
# ============================================================

# def tukey_fences(series, k=1.5):
#     """Return lower and upper Tukey fences."""
#     Q1 = series.quantile(0.25)
#     Q3 = series.quantile(0.75)
#     IQR = Q3 - Q1
#     return Q1 - k * IQR, Q3 + k * IQR
# 
# # Example: check outliers in a numeric column
# col = 'your_column'
# lower, upper = tukey_fences(df[col])
# outliers = df[(df[col] < lower) | (df[col] > upper)]
# print(f'{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

**Outlier strategy:** ___
(Trim? Winsorize? Keep with justification?)

### 2.6 Correlations

In [ ]:
# ============================================================
# 2.6 Correlation heatmap (Ch 3)
# ============================================================

# numeric_cols = df.select_dtypes(include=[np.number]).columns
# corr_matrix = df[numeric_cols].corr()
# 
# plt.figure(figsize=(10, 8))
# sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
# plt.title('Correlation Matrix')
# plt.tight_layout()
# plt.show()

### YOUR TASK: Answer These 3 EDA Questions

1. **What is the distribution of your target variable?** Is it balanced (classification) or normally distributed (regression)? If not, what will you do about it?

   *Your answer:* ___

2. **Which features appear most correlated with the target?** Are any features highly correlated with each other (multicollinearity)?  

   *Your answer:* ___

3. **What is the biggest data quality issue you found, and how will you handle it?**  

   *Your answer:* ___

### 2.7 Data Quality Summary

**Data Quality Summary**

My dataset has **N = ___** observations and **M = ___** features.

**Missing data:** ___% of cells are missing. The missingness pattern appears to be [MCAR / MAR / MNAR] because ___. I will handle missing data by ___.

**Outliers:** I identified ___ outliers using [Tukey Fences / IQR / visual inspection]. I will handle them by ___.

**Target variable:** [Describe distribution — class balance or spread].

**Key finding from EDA:** ___

---
## Part 3: Modeling

### 3.1 Train/Test Split

In [ ]:
# ============================================================
# 3.1 Train/test split (Ch 6)
# ============================================================

# # Define features and target
# X = df[['feature_1', 'feature_2', 'feature_3']]  # Replace with your features
# y = df['target']  # Replace with your target
# 
# # For classification: use stratify=y
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )
# 
# # For regression: remove stratify
# # X_train, X_test, y_train, y_test = train_test_split(
# #     X, y, test_size=0.2, random_state=RANDOM_STATE
# # )
# 
# print(f'Train: {X_train.shape[0]} samples')
# print(f'Test:  {X_test.shape[0]} samples')

### 3.2 Model 1: Baseline

In [ ]:
# ============================================================
# 3.2 Model 1 — Baseline
# ============================================================
# Choose a simple, interpretable baseline:
#   Classification: LogisticRegression
#   Regression: LinearRegression or Ridge

# from sklearn.linear_model import LogisticRegression  # or LinearRegression, Ridge
# 
# model_1 = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
# model_1.fit(X_train, y_train)
# 
# y_pred_1 = model_1.predict(X_test)
# 
# # Classification metrics
# print('Model 1: Logistic Regression')
# print(classification_report(y_test, y_pred_1))
# 
# # Regression metrics (use these instead if regression)
# # print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_1)):.4f}')
# # print(f'MAE:  {mean_absolute_error(y_test, y_pred_1):.4f}')
# # print(f'R2:   {r2_score(y_test, y_pred_1):.4f}')

### 3.3 Model 2: Your Choice

In [ ]:
# ============================================================
# 3.3 Model 2 — Your choice
# ============================================================
# Choose a more flexible model:
#   Classification: RandomForestClassifier, GradientBoostingClassifier
#   Regression: RandomForestRegressor, GradientBoostingRegressor

# from sklearn.ensemble import RandomForestClassifier  # or RandomForestRegressor
# 
# model_2 = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
# model_2.fit(X_train, y_train)
# 
# y_pred_2 = model_2.predict(X_test)
# 
# print('Model 2: Random Forest')
# print(classification_report(y_test, y_pred_2))

### 3.4 Cross-Validation Comparison

In [ ]:
# ============================================================
# 3.4 Cross-validation (Ch 15)
# ============================================================

# # Choose scoring: 'accuracy', 'f1', 'roc_auc' (classification)
# #                  'neg_mean_squared_error', 'r2' (regression)
# scoring = 'accuracy'
# 
# cv_1 = cross_val_score(model_1, X_train, y_train, cv=5, scoring=scoring)
# cv_2 = cross_val_score(model_2, X_train, y_train, cv=5, scoring=scoring)
# 
# print(f'Model 1 CV {scoring}: {cv_1.mean():.4f} +/- {cv_1.std():.4f}')
# print(f'Model 2 CV {scoring}: {cv_2.mean():.4f} +/- {cv_2.std():.4f}')
# 
# # Comparison table
# comparison = pd.DataFrame({
#     'Model': ['Model 1 (Baseline)', 'Model 2 (Your Choice)'],
#     f'CV {scoring} (mean)': [cv_1.mean(), cv_2.mean()],
#     f'CV {scoring} (std)': [cv_1.std(), cv_2.std()],
# })
# comparison

---
## Part 4: Feature Importance + Visualization

### 4.1 Feature Importance

In [ ]:
# ============================================================
# 4.1 Feature importance (Ch 19)
# ============================================================

# # For tree-based models:
# importances = pd.Series(
#     model_2.feature_importances_, index=X.columns
# ).sort_values(ascending=True)
# 
# fig, ax = plt.subplots(figsize=(8, 6))
# importances.plot(kind='barh', ax=ax, color='steelblue')
# ax.set_xlabel('Feature Importance (Gini)')
# ax.set_title('Feature Importance — Predictive, NOT Causal')
# 
# # CRITICAL: Add the caveat banner
# ax.text(
#     0.98, 0.02,
#     'Predictive importance only.\nDoes not imply causal effect.',
#     transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
#     style='italic', color='#c0392b',
#     bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdedec', edgecolor='#e74c3c')
# )
# 
# plt.tight_layout()
# plt.show()

### 4.2 Key Visualization for Your Report

In [ ]:
# ============================================================
# 4.2 Your key visualization
# ============================================================
# This is the ONE chart you would put on the first page of your report.
# It should communicate your main finding clearly.
#
# Examples:
#   - Actual vs. predicted scatter (regression)
#   - Confusion matrix heatmap (classification)
#   - ROC curve comparison (classification)
#   - Partial dependence plot for top feature

# YOUR CODE HERE

---
## Part 5: Recommendation

Use the SCR (Situation-Complication-Resolution) structure from Chapter 26.

**Situation:** ___
(What is the context? Who is the stakeholder? What decision do they face?)

**Complication:** ___
(What makes this decision hard? What uncertainty exists? What did your analysis reveal?)

**Resolution:** ___
(What do you recommend? Based on what evidence? With what confidence?)

**Uncertainty Statement:** Based on our cross-validation results (metric = ___ +/- ___), we estimate that ___. The primary limitation is ___. We recommend ___ with the caveat that ___.

---
## Part 6: Streamlit Export Guide

### 6.1 Creating app.py

Your Streamlit app should contain:
1. **Title and description** — `st.title()`, `st.markdown()`
2. **Input controls** — `st.slider()`, `st.selectbox()`, `st.number_input()`
3. **Model prediction** — load your trained model, generate predictions from user inputs
4. **Visualization** — at least one chart that updates with user inputs
5. **Uncertainty** — display confidence/prediction intervals alongside point estimates

### 6.2 Minimal app.py Template

```python
import streamlit as st
import pandas as pd
import numpy as np
import joblib  # to load saved model

st.title('Your Project Title')
st.markdown('Brief description of what this app predicts.')

# Sidebar controls
feature_1 = st.sidebar.slider('Feature 1', min_value=0.0, max_value=100.0, value=50.0)
feature_2 = st.sidebar.selectbox('Feature 2', ['Option A', 'Option B', 'Option C'])

# Load model (save with joblib.dump(model, 'model.pkl') in your notebook)
model = joblib.load('model.pkl')

# Predict
input_data = pd.DataFrame({'feature_1': [feature_1], 'feature_2': [feature_2]})
prediction = model.predict(input_data)[0]

st.metric('Prediction', f'{prediction:.2f}')
```

### 6.3 requirements.txt Template

```
streamlit>=1.31.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.4.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0
```

### 6.4 Deployment Steps

1. Save your model: `joblib.dump(model_2, 'model.pkl')`
2. Test locally: `streamlit run app.py`
3. Push to GitHub: `app.py`, `model.pkl`, `requirements.txt`
4. Go to [streamlit.io/cloud](https://streamlit.io/cloud) and deploy
5. Submit the permanent URL on Canvas

---
## Part 7: AI Methodology Appendix

Document at least **3 AI interactions** using the P.R.I.M.E. framework. Copy and fill in the template below for each interaction.

---

### AI Interaction 1

**Prep:** What did you need? What context did you have before prompting?
> ___

**Request:** What exact prompt did you write?
> ___

**Iterate:** What did the AI return? What did you change or refine?
> ___

**Mechanism Check:** How did you verify the output was correct?
> ___

**Evaluate:** What human judgment did you apply? What did you accept/reject and why?
> ___

---

### AI Interaction 2

**Prep:** ___

**Request:** ___

**Iterate:** ___

**Mechanism Check:** ___

**Evaluate:** ___

---

### AI Interaction 3

**Prep:** ___

**Request:** ___

**Iterate:** ___

**Mechanism Check:** ___

**Evaluate:** ___